# Data Preparation Lab -- Module 2, Class 2

**Dataset:** Superstore Sales

In this lab you will:
1. Load and inspect data
2. Handle missing values
3. Remove duplicates
4. Convert data types
5. Create derived features

The first 3 tasks are pre-built. The rest are TODO for you.

---
## Setup: Load the Dataset

Option A: Upload from Kaggle (download from https://www.kaggle.com/datasets/vivek468/superstore-dataset-final)

Option B: Use the URL loader below.

In [ ]:
import kagglehub
# Kagglehub is a specialized tool designed to interact with Kaggle's API.
# It handles the authentication and downloading process for you automatically.



# Download latest version
path = kagglehub.dataset_download("vivek468/superstore-dataset-final")
# We call the 'dataset_download' function.
# "vivek468/superstore-dataset-final" is the unique identifier for the dataset.
# The function connects to Kaggle, checks for the latest version, and downloads it.


print("Path to dataset files:", path)
# The download function returns a 'String' (the folder path on your computer).
# We print this path so we know exactly where to point our next tool (like Pandas).

Using Colab cache for faster access to the 'superstore-dataset-final' dataset.
Path to dataset files: /kaggle/input/superstore-dataset-final


In [ ]:
# Import Pandas for data tables and NumPy for numerical math
import pandas as pd
import numpy as np

# Option A: Upload in Colab
# from google.colab import files
# uploaded = files.upload()  # upload your CSV
# df = pd.read_csv('SampleSuperstore.csv')

# Option B: Load from a public URL
# If the URL does not work, use Option A.

# This URL points to a raw CSV file hosted on GitHub
url = "https://raw.githubusercontent.com/leonism/sample-superstore/master/data/superstore.csv"

# We use a 'try' block to handle potential connection issues
try:

    # Read the CSV from the internet.
    # 'latin-1' encoding helps read special characters without errors.
    df = pd.read_csv(url, encoding='latin-1')

    # .shape[0] gives the row count; .shape[1] gives the column count
    print(f"Loaded from URL: {df.shape[0]} rows, {df.shape[1]} columns")

# If the URL fails (e.g., no internet), this 'except' block runs
except Exception as e:
    print(f"URL failed ({e}). Use Option A: upload the CSV manually.")

    # Trigger a manual upload dialog in Google Colab
    # Fallback: upload manually
    from google.colab import files
    uploaded = files.upload()

    # Get the name of the file you just uploaded
    filename = list(uploaded.keys())[0]

    # Load that uploaded file into a DataFrame
    df = pd.read_csv(filename, encoding='latin-1')
    print(f"Loaded from upload: {df.shape[0]} rows, {df.shape[1]} columns")

Loaded from URL: 10800 rows, 21 columns


---
## Task 1: Inspect the Data (pre-built)

Always look at your data before doing anything to it.

In [ ]:
# Reads first 5 rows from dataset
# First 5 rows
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2017-152156,11/8/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2.0,0.00,41.9136
1,2,CA-2017-152156,11/8/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3.0,0.00,219.5820
2,3,CA-2017-138688,6/12/2017,6/16/2017,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2.0,0.00,6.8714
3,4,US-2016-108966,10/11/2016,10/18/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5.0,0.45,-383.0310
4,5,US-2016-108966,10/11/2016,10/18/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2.0,0.20,2.5164


In [ ]:
# Shape: rows x columns
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")

Shape: 10800 rows, 21 columns


In [ ]:
# Data types and non-null counts
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10800 entries, 0 to 10799
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         10800 non-null  object 
 1   Order ID       10800 non-null  object 
 2   Order Date     9994 non-null   object 
 3   Ship Date      9994 non-null   object 
 4   Ship Mode      9994 non-null   object 
 5   Customer ID    9994 non-null   object 
 6   Customer Name  9994 non-null   object 
 7   Segment        9994 non-null   object 
 8   Country        9994 non-null   object 
 9   City           9994 non-null   object 
 10  State          9994 non-null   object 
 11  Postal Code    9983 non-null   float64
 12  Region         9994 non-null   object 
 13  Product ID     9994 non-null   object 
 14  Category       9994 non-null   object 
 15  Sub-Category   9994 non-null   object 
 16  Product Name   9994 non-null   object 
 17  Sales          9994 non-null   float64
 18  Quanti

In [ ]:
# Summary statistics for numerical columns
df.describe()

---
## Task 2: Missing Values (pre-built)

Check which columns have missing values and how many.

In [ ]:
# Count missing values per column
# .isnull() creates a True/False map; .sum() counts the 'True' values per column
missing = df.isnull().sum()

# Divide null count by total rows (len(df)) to get the relative 'hole' size
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)

# Combine both metrics into a new Pandas DataFrame for a clear report [cite: 117, 121]
missing_report = pd.DataFrame({
    'missing_count': missing,
    'missing_pct': missing_pct
})

# Show only columns with missing values
# Use a boolean mask to show only columns that actually have missing data
missing_report[missing_report['missing_count'] > 0]

In [ ]:
# Fill missing numerical values with median (robust to outliers)
# Identify columns with numbers (integers or floats)
numerical_cols = df.select_dtypes(include=[np.number]).columns
for col in numerical_cols:

    # Check if the specific column has any "holes" (null values)
    if df[col].isnull().sum() > 0:

        # Use median because it is 'robust' (it isn't ruined by extremely high/low numbers)
        median_val = df[col].median()

        # fillna() replaces the holes with that median value
        df[col].fillna(median_val, inplace=True)
        print(f"Filled {col} missing values with median: {median_val}")

# Fill missing categorical values with mode
# Identify columns with text (objects)
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    if df[col].isnull().sum() > 0:

        # Use mode (the most frequently occurring item, like "Standard Class")
        mode_val = df[col].mode()[0]
        df[col].fillna(mode_val, inplace=True)
        print(f"Filled {col} missing values with mode: {mode_val}")

# Verify: no more missing values
# Count every null in the entire table to ensure it is now zero
print(f"\nTotal missing values remaining: {df.isnull().sum().sum()}")

---
## Task 3: Remove Duplicates (pre-built)

In [ ]:
# Check for duplicates
# .duplicated() marks repeated rows; .sum() counts how many 'True' repeats exist
n_duplicates = df.duplicated().sum()
print(f"Duplicate rows found: {n_duplicates}")

# Remove duplicates
# Only run the removal logic if duplicates actually exist [cite: 22, 29]
if n_duplicates > 0:

    # .drop_duplicates() deletes exact copies, keeping only the first instance
    df = df.drop_duplicates()

    # Verify the new dataset size after cleaning
    print(f"After removal: {df.shape[0]} rows remain")
else:

    # Provide feedback if the data was already clean [cite: 37]
    print("No duplicates to remove.")

---
## Task 4: Convert Date Columns (TODO)

The `Order Date` and `Ship Date` columns are stored as strings. Convert them to proper datetime objects.

Hint: Use `pd.to_datetime()`. After conversion, verify with `.dtypes`.

In [ ]:
# Loop through every column name in the dataset
for col in df.columns:

    # Check if the word 'date' (any case) is in the column name [cite: 3]
    if 'date' in col.lower() or 'Date' in col:

        # Print the data type (e.g., 'object' vs 'datetime') [cite: 142]
        print(f"  {col}: {df[col].dtype}")

        # Show the first actual value to see the current format (MM/DD/YYYY, etc.) [cite: 13]
        print(f"  Sample value: {df[col].iloc[0]}")

In [ ]:
# TODO: Convert date columns to datetime
# Look at the column names printed above and convert them.
# The column names may vary depending on the dataset version.
#
# Example pattern:
# df['Column Name'] = pd.to_datetime(df['Column Name'])

# Your code here:


In [ ]:
# TODO: Verify the conversion worked
# Print dtypes for the date columns to confirm they are datetime64

# Your code here:


---
## Task 5: Derived Features (TODO)

Create customer-level summary features. These are the building blocks for customer segmentation (Activity 4).

You need to create:
- **total_spending**: Total sales per customer
- **order_frequency**: Number of orders per customer
- **avg_order_value**: Average sales amount per order per customer

Hint: Use `df.groupby('Customer ID')` (or whatever the customer ID column is named).

In [ ]:
# First, identify the right column names
print("All columns:")
for col in df.columns:
    print(f"  {col}")

In [ ]:
# TODO: Create total_spending per customer
# Hint: df.groupby('Customer ID')['Sales'].sum()
#
# Replace column names below with the actual names from your dataset.

# customer_spending = df.groupby('???')['???'].sum()
# customer_spending.name = 'total_spending'

# Your code here:


In [ ]:
# TODO: Create order_frequency per customer
# Hint: Count the number of rows (orders) per customer.
# Use .groupby(...).size() or .groupby(...)['some_col'].count()

# Your code here:


In [ ]:
# TODO: Create avg_order_value per customer
# Hint: Use .groupby(...)['Sales'].mean()

# Your code here:


In [ ]:
# TODO: Combine all three into a single customer-level DataFrame
# Hint: Use pd.concat([series1, series2, series3], axis=1)
# or create them all at once with .groupby(...).agg(...)

# customer_summary = pd.concat([customer_spending, order_freq, avg_order], axis=1)
# customer_summary.columns = ['total_spending', 'order_frequency', 'avg_order_value']

# Your code here:


In [ ]:
# TODO: Display the first 10 rows of your customer summary and .describe()

# Your code here:


---
## Task 6: Save Cleaned Data (TODO)

Save the cleaned DataFrame to a new CSV file. Never overwrite the original.

In [ ]:
# TODO: Save the cleaned main DataFrame
# df.to_csv('superstore_cleaned.csv', index=False)

# TODO: Save the customer summary DataFrame
# customer_summary.to_csv('customer_summary.csv')

# Your code here:


---
## Reflection

Answer these in a text cell or comments:

1. Why did we use median instead of mean for filling numerical missing values?
2. What is the difference between the row-level DataFrame (one row per order) and the customer-level summary? When would you use each?
3. If two rows have identical values in every column, are they always true duplicates? When might they not be?